# SSE Association Sparsity and Separation Diagnostics

This notebook checks whether the association-analysis model frames are sparse or nearly separated. The global candidate/background balance can look healthy while individual windows, clades, predictor levels, or complete-case model frames still contain only one outcome class or very small minority-class counts.

The checks below use the same prepared frames as the association pipeline.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate config.yaml from current path.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection import lib as sselib  # noqa: E402

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

## Configuration

In [2]:
WINDOW_STRIDE = 2
VARIANT_ADJUSTER = "clade"
SPARSE_MIN_CLASS = 10
NEAR_DETERMINISTIC_RATE = 0.98
N_BINS = 10

MODEL_SETS = sselib.default_model_sets(
    variant_adjuster=VARIANT_ADJUSTER,
    window_adjustment="fixed_effects",
)

MIXING_FEATURES = list(sselib.DEFAULT_MIXING_FEATURES)
COMPOSITION_FEATURES = [spec["column"] for spec in sselib.COMPOSITION_SPECS]

MODEL_SETS, MIXING_FEATURES, COMPOSITION_FEATURES

({'primary': ['C(window_idx)', 'C(clade)'],
  'expanded': ['C(window_idx)',
   'C(clade)',
   'z_dz_cum_prop_sequenced',
   'z_dz_cum_incidence_per_capita',
   'z_dz_7d_test_positivity',
   'z_log1p_dz_cum_positive_tests']},
 ['sex_entropy_z',
  'age_entropy_z',
  'simd_entropy_z',
  'urban_rural_entropy_z',
  'health_board_entropy_z'],
 ['sex',
  'age_band',
  'dz_simd_quintile',
  'dz_urban_rural_class',
  'dz_health_board'])

## Load Analysis Frames

In [3]:
frames = sselib.load_association_frames(
    project_root=PROJECT_ROOT,
    variant_adjuster=VARIANT_ADJUSTER,
    group_by_clade=False,
    window_stride=WINDOW_STRIDE,
    run_composition=True,
)

node_df = frames.node_model_base.copy()
composition_df = frames.composition_base.copy()

display(
    pd.DataFrame(
        [
            {
                "frame": "node_model_base",
                "rows": len(node_df),
                "nodes": node_df["cluster_id"].nunique(),
                "background": int((node_df["candidate"] == 0).sum()),
                "candidate": int((node_df["candidate"] == 1).sum()),
            },
            {
                "frame": "composition_base",
                "rows": len(composition_df),
                "nodes": composition_df["cluster_id"].nunique(),
                "background": int((composition_df["candidate"] == 0).sum()),
                "candidate": int((composition_df["candidate"] == 1).sum()),
            },
        ]
    )
)

print(f"Minimum candidate cluster size threshold: {frames.min_candidate_size}")
display(frames.cluster_diagnostics)

,frame,rows,nodes,background,candidate
0,node_model_base,13059,13059,6152,6907
1,composition_base,264139,13059,116683,147456


Minimum candidate cluster size threshold: 6


,cluster_col,n_rows,n_clusters,min_rows_per_cluster,median_rows_per_cluster,outcome_positive_clusters,outcome_varying_clusters,analysis_frame
0,cluster_id,264139,13059,1,9.0,6907,0,composition
1,cluster_id,13059,13059,1,1.0,6907,0,node_mixing


## Helper Functions

In [4]:
def normalise_group_cols(group_cols):
    return [group_cols] if isinstance(group_cols, str) else list(group_cols)


def outcome_balance(data, outcome="candidate"):
    counts = data[outcome].dropna().astype(int).value_counts().reindex([0, 1], fill_value=0)
    total = int(counts.sum())
    return pd.DataFrame(
        [
            {
                "background": int(counts.loc[0]),
                "candidate": int(counts.loc[1]),
                "total": total,
                "candidate_rate": counts.loc[1] / total if total else np.nan,
                "min_class": int(counts.min()) if total else 0,
            }
        ]
    )


def balance_by(
    data,
    group_cols,
    *,
    outcome="candidate",
    min_class_threshold=SPARSE_MIN_CLASS,
    near_rate=NEAR_DETERMINISTIC_RATE,
):
    group_cols = normalise_group_cols(group_cols)
    required = [*group_cols, outcome]
    d = data.dropna(subset=required).copy()
    if d.empty:
        return pd.DataFrame()

    d[outcome] = d[outcome].astype(int)
    counts = d.groupby([*group_cols, outcome], dropna=False).size().unstack(outcome, fill_value=0)
    for cls in [0, 1]:
        if cls not in counts.columns:
            counts[cls] = 0
    counts = counts[[0, 1]].rename(columns={0: "background", 1: "candidate"})
    out = counts.reset_index()
    out["total"] = out["background"] + out["candidate"]
    out["candidate_rate"] = np.where(out["total"].gt(0), out["candidate"] / out["total"], np.nan)
    out["min_class"] = out[["background", "candidate"]].min(axis=1)
    out["separated"] = out["min_class"].eq(0)
    out["sparse"] = out["min_class"].lt(min_class_threshold)
    out["near_deterministic"] = out["candidate_rate"].ge(near_rate) | out["candidate_rate"].le(1 - near_rate)
    return out.sort_values(
        ["separated", "sparse", "min_class", "total"],
        ascending=[False, False, True, False],
    ).reset_index(drop=True)


def summarise_balance(table, name):
    if table.empty:
        return {
            "check": name,
            "groups": 0,
            "separated_groups": 0,
            "sparse_groups": 0,
            "near_deterministic_groups": 0,
            "min_class_min": np.nan,
        }
    return {
        "check": name,
        "groups": len(table),
        "separated_groups": int(table["separated"].sum()),
        "sparse_groups": int(table["sparse"].sum()),
        "near_deterministic_groups": int(table["near_deterministic"].sum()),
        "min_class_min": int(table["min_class"].min()),
    }


def binned_balance(data, feature, *, q=N_BINS, outcome="candidate"):
    d = data[[feature, outcome]].dropna().copy()
    if d[feature].nunique(dropna=True) < 2:
        return pd.DataFrame()
    d[f"{feature}_bin"] = pd.qcut(d[feature], q=q, duplicates="drop")
    out = balance_by(d, f"{feature}_bin", outcome=outcome)
    out.insert(0, "feature", feature)
    return out


def complete_case_frame(data, predictors, adjusters):
    required = ["candidate", *predictors, *sselib.model_variables_from_terms(adjusters)]
    required = list(dict.fromkeys(required))
    missing = [col for col in required if col not in data.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    return data.dropna(subset=required).copy()


def model_case_summary(data, *, domain, model_set, predictor_set, predictors, adjusters, group_checks):
    d = complete_case_frame(data, predictors, adjusters)
    overall = outcome_balance(d).iloc[0]
    row = {
        "domain": domain,
        "model_set": model_set,
        "predictor_set": predictor_set,
        "predictors": "+".join(predictors),
        "n_rows": len(d),
        "background": int(overall["background"]),
        "candidate": int(overall["candidate"]),
        "candidate_rate": overall["candidate_rate"],
        "min_class": int(overall["min_class"]),
    }
    for check_name, group_cols in group_checks.items():
        group_cols = normalise_group_cols(group_cols)
        if not all(col in d.columns for col in group_cols):
            row[f"{check_name}_groups"] = np.nan
            row[f"{check_name}_separated"] = np.nan
            row[f"{check_name}_sparse"] = np.nan
            row[f"{check_name}_min_class"] = np.nan
            continue
        table = balance_by(d, group_cols)
        row[f"{check_name}_groups"] = len(table)
        row[f"{check_name}_separated"] = int(table["separated"].sum()) if not table.empty else 0
        row[f"{check_name}_sparse"] = int(table["sparse"].sum()) if not table.empty else 0
        row[f"{check_name}_min_class"] = int(table["min_class"].min()) if not table.empty else np.nan
    return row

## Overall Outcome Balance

In [5]:
display(outcome_balance(node_df).assign(frame="node_model_base"))
display(outcome_balance(composition_df).assign(frame="composition_base"))

,background,candidate,total,candidate_rate,min_class,frame
0,6152,6907,13059,0.528907,6152,node_model_base


,background,candidate,total,candidate_rate,min_class,frame
0,116683,147456,264139,0.558252,116683,composition_base


## Node-Level Strata Checks

These are the most relevant diagnostics for node-level mixing models. `window_x_clade` is not an explicit interaction in the default formula, but it is useful for spotting local data gaps.

In [6]:
node_strata_checks = {
    "window": "window_idx",
    "clade": "clade",
    "window_x_clade": ["window_idx", "clade"],
}

node_strata_tables = {}
node_strata_summary = []

for name, group_cols in node_strata_checks.items():
    table = balance_by(node_df, group_cols)
    node_strata_tables[name] = table
    node_strata_summary.append(summarise_balance(table, name))
    print(f"\n{name}")
    display(table.head(25))

node_strata_summary = pd.DataFrame(node_strata_summary)
display(node_strata_summary)


window


candidate,window_idx,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,1,0,1,1,1.000000,0,True,True,True
1,2,2,1,3,0.333333,1,False,True,False
2,3,3,10,13,0.769231,3,False,True,False
3,63,10,26,36,0.722222,10,False,False,False
4,4,11,10,21,0.476190,10,False,False,False
5,64,11,41,52,0.788462,11,False,False,False
6,61,12,30,42,0.714286,12,False,False,False
7,66,12,28,40,0.700000,12,False,False,False
8,67,12,15,27,0.555556,12,False,False,False
9,5,13,19,32,0.593750,13,False,False,False



clade


candidate,clade,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,22F,0,3,3,1.000000,0,True,True,True
1,20D,0,1,1,1.000000,0,True,True,True
2,21D,1,0,1,0.000000,0,True,True,True
3,19B,1,1,2,0.500000,1,False,True,False
4,23C,5,14,19,0.736842,5,False,True,False
5,23A,6,13,19,0.684211,6,False,True,False
6,21A,8,7,15,0.466667,7,False,True,False
7,22D,10,23,33,0.696970,10,False,False,False
8,20A,18,19,37,0.513514,18,False,False,False
9,recombinant,19,20,39,0.512821,19,False,False,False



window_x_clade


candidate,window_idx,clade,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,47,22C,0,5,5,1.0,0,True,True,True
1,24,21I,0,4,4,1.0,0,True,True,True
2,3,20A,0,3,3,1.0,0,True,True,True
3,3,20E,0,3,3,1.0,0,True,True,True
4,22,21J,0,3,3,1.0,0,True,True,True
5,37,21I,0,3,3,1.0,0,True,True,True
6,40,21L,0,3,3,1.0,0,True,True,True
7,49,22A,0,3,3,1.0,0,True,True,True
8,58,22A,0,3,3,1.0,0,True,True,True
9,59,22D,0,3,3,1.0,0,True,True,True


,check,groups,separated_groups,sparse_groups,near_deterministic_groups,min_class_min
0,window,67,1,3,1,0
1,clade,21,3,7,3,0
2,window_x_clade,206,55,141,55,0


## Clade-Group Sensitivity Balance

In [7]:
node_with_clade_group = sselib.add_clade_group(node_df, source_col="clade", target_col="clade_group")
clade_group_balance = balance_by(node_with_clade_group, "clade_group")
display(clade_group_balance)

clade_window_balance = balance_by(node_with_clade_group, ["clade_group", "window_idx"])
display(clade_window_balance.head(40))

candidate,clade_group,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,20A,18,19,37,0.513514,18,False,False,False
1,21I (Delta),25,24,49,0.489796,24,False,False,False
2,22C (Omicron),26,31,57,0.543860,26,False,False,False
3,20B,33,28,61,0.459016,28,False,False,False
4,Other,50,82,132,0.621212,50,False,False,False
5,22A (Omicron),55,70,125,0.560000,55,False,False,False
6,22E (Omicron),56,140,196,0.714286,56,False,False,False
7,20E (EU1),174,172,346,0.497110,172,False,False,False
8,22B (Omicron),397,554,951,0.582545,397,False,False,False
9,20I (Alpha),884,625,1509,0.414182,625,False,False,False


candidate,clade_group,window_idx,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,22C (Omicron),47,0,5,5,1.0,0,True,True,True
1,21I (Delta),24,0,4,4,1.0,0,True,True,True
2,20A,3,0,3,3,1.0,0,True,True,True
3,20E (EU1),3,0,3,3,1.0,0,True,True,True
4,21I (Delta),37,0,3,3,1.0,0,True,True,True
5,21J (Delta),22,0,3,3,1.0,0,True,True,True
6,21L (Omicron),40,0,3,3,1.0,0,True,True,True
7,22A (Omicron),49,0,3,3,1.0,0,True,True,True
8,22A (Omicron),58,0,3,3,1.0,0,True,True,True
9,Other,59,0,3,3,1.0,0,True,True,True


## Composition Predictor Level Checks

These checks use the sequence-level composition frame. The outcome is still the cluster/node-level candidate label.

In [8]:
composition_level_tables = []
composition_level_summary = []

for spec in sselib.COMPOSITION_SPECS:
    col = spec["column"]
    table = balance_by(composition_df, col)
    table.insert(0, "predictor", spec["name"])
    composition_level_tables.append(table)
    composition_level_summary.append(summarise_balance(table, spec["name"]))
    print(f"\n{spec['label']} ({col})")
    display(table.head(25))

composition_level_balance = pd.concat(composition_level_tables, ignore_index=True)
composition_level_summary = pd.DataFrame(composition_level_summary)
display(composition_level_summary)


Sex (sex)


candidate,predictor,sex,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,sex,Male,55357,69134,124491,0.555333,55357,False,False,False
1,sex,Female,61326,78322,139648,0.560853,61326,False,False,False



Age band (age_band)


candidate,predictor,age_band,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,age_band,70-74,2480,3290,5770,0.570191,2480,False,False,False
1,age_band,00-04,2695,3117,5812,0.536304,2695,False,False,False
2,age_band,65-69,2982,3931,6913,0.568639,2982,False,False,False
3,age_band,60-64,5300,7004,12304,0.569246,5300,False,False,False
4,age_band,75+,6246,8417,14663,0.574030,6246,False,False,False
5,age_band,05-09,7069,7871,14940,0.526841,7069,False,False,False
6,age_band,55-59,7181,9325,16506,0.564946,7181,False,False,False
7,age_band,45-49,7771,9790,17561,0.557485,7771,False,False,False
8,age_band,50-54,7896,10281,18177,0.565605,7896,False,False,False
9,age_band,10-14,8688,9710,18398,0.527775,8688,False,False,False



SIMD quintile (dz_simd_quintile)


...


Urban/rural class (dz_urban_rural_class)


candidate,predictor,dz_urban_rural_class,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,urban_rural_class,Remote Small Towns,3148,4189,7337,0.570942,3148,False,False,False
1,urban_rural_class,Remote Rural,3883,5220,9103,0.573437,3883,False,False,False
2,urban_rural_class,Accessible Small Towns,9505,11950,21455,0.556980,9505,False,False,False
3,urban_rural_class,Accessible Rural,11213,14239,25452,0.559445,11213,False,False,False
4,urban_rural_class,Other Urban Areas,44232,56215,100447,0.559648,44232,False,False,False
5,urban_rural_class,Large Urban Areas,44702,55643,100345,0.554517,44702,False,False,False



Health board (dz_health_board)


candidate,predictor,dz_health_board,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,health_board,Western Isles,119,198,317,0.624606,119,False,False,False
1,health_board,Shetland,334,497,831,0.598075,334,False,False,False
2,health_board,Orkney,341,514,855,0.601170,341,False,False,False
3,health_board,Borders,1947,2297,4244,0.541235,1947,False,False,False
4,health_board,Dumfries and Galloway,2484,3147,5631,0.558871,2484,False,False,False
5,health_board,Highland,5184,6784,11968,0.566845,5184,False,False,False
6,health_board,Forth Valley,7183,8918,16101,0.553879,7183,False,False,False
7,health_board,Fife,7761,9794,17555,0.557904,7761,False,False,False
8,health_board,Ayrshire and Arran,7840,10280,18120,0.567329,7840,False,False,False
9,health_board,Tayside,8613,11090,19703,0.562858,8613,False,False,False


,check,groups,separated_groups,sparse_groups,near_deterministic_groups,min_class_min
0,sex,2,0,0,0,55357
1,age_band,16,0,0,0,2480
2,simd_quintile,5,0,0,0,20923
3,urban_rural_class,6,0,0,0,3148
4,health_board,14,0,0,0,119


## Mixing Predictor Decile Checks

Continuous predictors cannot be checked by exact levels. Binning them into deciles gives a practical screen for near-deterministic outcome regions.

In [9]:
mixing_bin_tables = []
mixing_bin_summary = []

for feature in MIXING_FEATURES:
    table = binned_balance(node_df, feature)
    mixing_bin_tables.append(table)
    mixing_bin_summary.append(summarise_balance(table, feature))
    print(f"\n{feature}")
    display(table)

mixing_bin_balance = pd.concat(mixing_bin_tables, ignore_index=True)
mixing_bin_summary = pd.DataFrame(mixing_bin_summary)
display(mixing_bin_summary)


sex_entropy_z


candidate,feature,sex_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,sex_entropy_z,"(0.341, 0.503]",585,671,1256,0.534236,585,False,False,False
1,sex_entropy_z,"(-53.836999999999996, -1.402]",592,705,1297,0.543562,592,False,False,False
2,sex_entropy_z,"(-0.574, -0.135]",603,700,1303,0.537222,603,False,False,False
3,sex_entropy_z,"(-1.402, -0.574]",606,697,1303,0.534919,606,False,False,False
4,sex_entropy_z,"(0.602, 0.648]",606,695,1301,0.534204,606,False,False,False
5,sex_entropy_z,"(0.648, 0.693]",610,681,1291,0.527498,610,False,False,False
6,sex_entropy_z,"(-0.135, 0.191]",626,658,1284,0.512461,626,False,False,False
7,sex_entropy_z,"(0.503, 0.602]",627,670,1297,0.516577,627,False,False,False
8,sex_entropy_z,"(0.693, 1.828]",628,668,1296,0.515432,628,False,False,False
9,sex_entropy_z,"(0.191, 0.341]",654,685,1339,0.511576,654,False,False,False



age_entropy_z


candidate,feature,age_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,age_entropy_z,"(-25.427, -2.803]",565,732,1297,0.564379,565,False,False,False
1,age_entropy_z,"(0.114, 0.479]",573,718,1291,0.556158,573,False,False,False
2,age_entropy_z,"(-1.845, -1.28]",599,700,1299,0.538876,599,False,False,False
3,age_entropy_z,"(0.943, 3.116]",614,683,1297,0.526600,614,False,False,False
4,age_entropy_z,"(-0.492, -0.172]",614,682,1296,0.526235,614,False,False,False
5,age_entropy_z,"(0.479, 0.943]",628,669,1297,0.515806,628,False,False,False
6,age_entropy_z,"(-2.803, -1.845]",629,668,1297,0.515035,629,False,False,False
7,age_entropy_z,"(-1.28, -0.886]",631,663,1294,0.512365,631,False,False,False
8,age_entropy_z,"(-0.886, -0.492]",634,663,1297,0.511180,634,False,False,False
9,age_entropy_z,"(-0.172, 0.114]",650,652,1302,0.500768,650,False,False,False



simd_entropy_z


...


urban_rural_entropy_z


candidate,feature,urban_rural_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,urban_rural_entropy_z,"(0.735, 5.277]",571,723,1294,0.558733,571,False,False,False
1,urban_rural_entropy_z,"(-1.507, -1.073]",594,703,1297,0.542020,594,False,False,False
2,urban_rural_entropy_z,"(0.154, 0.735]",599,701,1300,0.539231,599,False,False,False
3,urban_rural_entropy_z,"(-0.652, -0.264]",599,698,1297,0.538165,599,False,False,False
4,urban_rural_entropy_z,"(-1.073, -0.652]",599,697,1296,0.537809,599,False,False,False
5,urban_rural_entropy_z,"(-2.607, -1.974]",604,693,1297,0.534310,604,False,False,False
6,urban_rural_entropy_z,"(-0.264, 0.154]",631,665,1296,0.513117,631,False,False,False
7,urban_rural_entropy_z,"(-12.577, -3.569]",634,663,1297,0.511180,634,False,False,False
8,urban_rural_entropy_z,"(-3.569, -2.607]",663,635,1298,0.489214,635,False,False,False
9,urban_rural_entropy_z,"(-1.974, -1.507]",643,652,1295,0.503475,643,False,False,False



health_board_entropy_z


candidate,feature,health_board_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,health_board_entropy_z,"(-0.632, 3.935]",556,741,1297,0.571318,556,False,False,False
1,health_board_entropy_z,"(-3.001, -2.329]",597,702,1299,0.540416,597,False,False,False
2,health_board_entropy_z,"(-2.329, -1.547]",607,690,1297,0.531997,607,False,False,False
3,health_board_entropy_z,"(-1.547, -0.632]",612,682,1294,0.527048,612,False,False,False
4,health_board_entropy_z,"(-6.184, -5.265]",616,670,1286,0.520995,616,False,False,False
5,health_board_entropy_z,"(-30.562, -7.611]",619,678,1297,0.522745,619,False,False,False
6,health_board_entropy_z,"(-7.611, -6.184]",621,686,1307,0.524866,621,False,False,False
7,health_board_entropy_z,"(-4.466, -3.667]",621,676,1297,0.521203,621,False,False,False
8,health_board_entropy_z,"(-3.667, -3.001]",630,666,1296,0.513889,630,False,False,False
9,health_board_entropy_z,"(-5.265, -4.466]",658,639,1297,0.492675,639,False,False,False


,check,groups,separated_groups,sparse_groups,near_deterministic_groups,min_class_min
0,sex_entropy_z,10,0,0,0,585
1,age_entropy_z,10,0,0,0,565
2,simd_entropy_z,10,0,0,0,575
3,urban_rural_entropy_z,10,0,0,0,571
4,health_board_entropy_z,10,0,0,0,556


## Complete-Case Model Frame Checks

These rows summarize the complete-case data available to each default main-analysis model. They are a compact way to see whether adding adjusters creates separated or sparse fixed-effect strata.

In [10]:
model_rows = []
node_group_checks = {
    "window": "window_idx",
    "clade": "clade",
    "window_x_clade": ["window_idx", "clade"],
}
composition_group_checks = {
    "window": "window_idx",
    "clade": "clade",
    "window_x_clade": ["window_idx", "clade"],
}

for model_set, adjusters in MODEL_SETS.items():
    for feature in MIXING_FEATURES:
        model_rows.append(
            model_case_summary(
                node_df,
                domain="node_mixing",
                model_set=model_set,
                predictor_set="single",
                predictors=[feature],
                adjusters=adjusters,
                group_checks=node_group_checks,
            )
        )
    model_rows.append(
        model_case_summary(
            node_df,
            domain="node_mixing",
            model_set=model_set,
            predictor_set="joint",
            predictors=MIXING_FEATURES,
            adjusters=adjusters,
            group_checks=node_group_checks,
        )
    )

    for spec in sselib.COMPOSITION_SPECS:
        model_rows.append(
            model_case_summary(
                composition_df,
                domain="composition",
                model_set=model_set,
                predictor_set="single",
                predictors=[spec["column"]],
                adjusters=adjusters,
                group_checks=composition_group_checks,
            )
        )
    model_rows.append(
        model_case_summary(
            composition_df,
            domain="composition",
            model_set=model_set,
            predictor_set="joint",
            predictors=COMPOSITION_FEATURES,
            adjusters=adjusters,
            group_checks=composition_group_checks,
        )
    )

model_complete_case_summary = pd.DataFrame(model_rows)
display(
    model_complete_case_summary.sort_values(
        ["domain", "model_set", "predictor_set", "predictors"]
    )
)

,domain,model_set,predictor_set,predictors,n_rows,background,candidate,candidate_rate,min_class,window_groups,window_separated,window_sparse,window_min_class,clade_groups,clade_separated,clade_sparse,clade_min_class,window_x_clade_groups,window_x_clade_separated,window_x_clade_sparse,window_x_clade_min_class
23,composition,expanded,joint,sex+age_band+dz_simd_quintile+dz_urban_rural_c...,264119,116676,147443,0.558245,116676,67,1,1,0,21,3,4,0,206,55,78,0
19,composition,expanded,single,age_band,264119,116676,147443,0.558245,116676,67,1,1,0,21,3,4,0,206,55,78,0
22,composition,expanded,single,dz_health_board,264119,116676,147443,0.558245,116676,67,1,1,0,21,3,4,0,206,55,78,0
20,composition,expanded,single,dz_simd_quintile,264119,116676,147443,0.558245,116676,67,1,1,0,21,3,4,0,206,55,78,0
21,composition,expanded,single,dz_urban_rural_class,264119,116676,147443,0.558245,116676,67,1,1,0,21,3,4,0,206,55,78,0
18,composition,expanded,single,sex,264119,116676,147443,0.558245,116676,67,1,1,0,21,3,4,0,206,55,78,0
11,composition,primary,joint,sex+age_band+dz_simd_quintile+dz_urban_rural_c...,264139,116683,147456,0.558252,116683,67,1,1,0,21,3,4,0,206,55,78,0
7,composition,primary,single,age_band,264139,116683,147456,0.558252,116683,67,1,1,0,21,3,4,0,206,55,78,0
10,composition,primary,single,dz_health_board,264139,116683,147456,0.558252,116683,67,1,1,0,21,3,4,0,206,55,78,0
8,composition,primary,single,dz_simd_quintile,264139,116683,147456,0.558252,116683,67,1,1,0,21,3,4,0,206,55,78,0


## Optional Standard-GLM Stress Test

Firth models are used because they are more stable under sparse or separated data. This optional ordinary-GLM fit is a diagnostic: very large coefficients, very large standard errors, warnings, or fitted probabilities near 0/1 indicate separation pressure.

In [11]:
OPTIONAL_FORMULA = "candidate ~ sex_entropy_z + C(window_idx) + C(clade)"
OPTIONAL_REQUIRED = ["candidate", "sex_entropy_z", "window_idx", "clade"]

stress_df = node_df.dropna(subset=OPTIONAL_REQUIRED).copy()
print(f"Rows in stress-test model frame: {len(stress_df):,}")

try:
    stress_result = smf.glm(
        OPTIONAL_FORMULA,
        data=stress_df,
        family=sm.families.Binomial(),
        missing="raise",
    ).fit(maxiter=100)

    stress_terms = pd.DataFrame(
        {
            "estimate": stress_result.params,
            "std_error": stress_result.bse,
            "z": stress_result.tvalues,
            "p_value": stress_result.pvalues,
        }
    )
    pred = stress_result.predict(stress_df)
    print(
        "Near-zero/one fitted probability share:",
        float(((pred < 1e-6) | (pred > 1 - 1e-6)).mean()),
    )
    display(stress_terms.assign(abs_estimate=stress_terms["estimate"].abs()).sort_values("abs_estimate", ascending=False).head(20))
    display(stress_terms.sort_values("std_error", ascending=False).head(20))
except Exception as exc:
    print(f"Ordinary GLM stress test failed: {type(exc).__name__}: {exc}")

Rows in stress-test model frame: 12,967
Near-zero/one fitted probability share: 0.0004627130407958664


,estimate,std_error,z,p_value,abs_estimate
C(window_idx)[T.67],-29.850108,29233.086006,-0.001021,0.999185,29.850108
C(window_idx)[T.53],-29.797370,29233.086001,-0.001019,0.999187,29.797370
C(window_idx)[T.62],-29.344245,29233.086003,-0.001004,0.999199,29.344245
C(window_idx)[T.66],-29.315065,29233.086004,-0.001003,0.999200,29.315065
C(window_idx)[T.65],-29.209272,29233.086004,-0.000999,0.999203,29.209272
C(window_idx)[T.57],-29.202265,29233.086003,-0.000999,0.999203,29.202265
C(window_idx)[T.54],-29.167116,29233.086002,-0.000998,0.999204,29.167116
C(window_idx)[T.55],-29.093890,29233.086002,-0.000995,0.999206,29.093890
C(window_idx)[T.56],-29.087876,29233.086003,-0.000995,0.999206,29.087876
C(window_idx)[T.60],-29.049815,29233.086003,-0.000994,0.999207,29.049815


,estimate,std_error,z,p_value
Intercept,23.390759,29233.086031,0.000800,0.999362
C(window_idx)[T.2],-22.270560,29233.086019,-0.000762,0.999392
C(window_idx)[T.67],-29.850108,29233.086006,-0.001021,0.999185
C(window_idx)[T.66],-29.315065,29233.086004,-0.001003,0.999200
C(window_idx)[T.64],-28.630364,29233.086004,-0.000979,0.999219
C(window_idx)[T.63],-28.875522,29233.086004,-0.000988,0.999212
C(window_idx)[T.65],-29.209272,29233.086004,-0.000999,0.999203
C(window_idx)[T.61],-28.758278,29233.086004,-0.000984,0.999215
C(window_idx)[T.62],-29.344245,29233.086003,-0.001004,0.999199
C(window_idx)[T.60],-29.049815,29233.086003,-0.000994,0.999207


## Export Diagnostics

In [12]:
DIAGNOSTIC_DIR = PROJECT_ROOT / "sse_detection" / "results" / "separation_diagnostics"
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

node_strata_summary_df = (
    node_strata_summary
    if isinstance(node_strata_summary, pd.DataFrame)
    else pd.DataFrame(node_strata_summary)
)
node_strata_summary_df.to_csv(DIAGNOSTIC_DIR / "node_strata_summary.csv", index=False)
pd.concat(
    [table.assign(check=name) for name, table in node_strata_tables.items()],
    ignore_index=True,
).to_csv(DIAGNOSTIC_DIR / "node_strata_balance.csv", index=False)
clade_group_balance.to_csv(DIAGNOSTIC_DIR / "clade_group_balance.csv", index=False)
clade_window_balance.to_csv(DIAGNOSTIC_DIR / "clade_window_balance.csv", index=False)
composition_level_summary.to_csv(DIAGNOSTIC_DIR / "composition_level_summary.csv", index=False)
composition_level_balance.to_csv(DIAGNOSTIC_DIR / "composition_level_balance.csv", index=False)
mixing_bin_summary.to_csv(DIAGNOSTIC_DIR / "mixing_bin_summary.csv", index=False)
mixing_bin_balance.to_csv(DIAGNOSTIC_DIR / "mixing_bin_balance.csv", index=False)
model_complete_case_summary.to_csv(DIAGNOSTIC_DIR / "model_complete_case_summary.csv", index=False)

print(f"Saved diagnostics to: {DIAGNOSTIC_DIR}")

Saved diagnostics to: /Users/ydnkka/Desktop/PhD Project/projects/scotland/sse_detection/results/separation_diagnostics
